# 02a — Digitization Style Comparison: Hank vs Tanna

Compares road digitization output between two participants (Hank and Tanna) for Palabek settlement.
Analysis is scoped to the **22 grid cells** that appear in both participants' assignment files with `is_overlap=True`,
i.e. cells that were intentionally assigned to both for QA purposes.

**Metrics computed per cell:**
- Total road length (m)
- Road density (m / km²)
- Number of line features (segments)
- Number of nodes (vertices across all geometries)
- Average segment length (m)
- Geometric overlap between the two digitizations (buffered intersection / union)

**Data:**
| Item | Path |
|---|---|
| Hank digitization | `data/processed/sample_digitization/Lawit-S2-digitize.geojson` |
| Tanna digitization | `data/processed/sample_digitization/Sample_Digitization_Palabek_Tanna.geojson` |
| Hank assignments | `data/processed/assignments/palabek_1km_road_intersect_hank_assignments.csv` |
| Tanna assignments | `data/processed/assignments/palabek_1km_road_intersect_tanna_assignments.csv` |
| Grid cells | `data/processed/palabek_1km_grids.geojson` |

In [1]:
# ── Parameters ────────────────────────────────────────────────────────────────
PROJECTED_CRS     = "EPSG:32636"   # UTM Zone 36N
CELL_SIZE_M       = 1000           # grid cell side in metres
SNAP_BUFFER_M     = 5              # buffer for geometric overlap calculation
N_EXAMPLE_CELLS   = 6              # number of individual cells to plot side-by-side

In [2]:
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import yaml
from scipy import stats
from shapely.ops import unary_union

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.1f}".format)

In [3]:
def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "paths.yaml").exists():
            return candidate
    raise FileNotFoundError("configs/paths.yaml not found")

project_root = find_project_root()
with open(project_root / "configs" / "paths.yaml") as f:
    paths = yaml.safe_load(f)

processed_dir = project_root / paths["data"]["processed"]
print(f"Project root : {project_root}")
print(f"Processed dir: {processed_dir}")

Project root : C:\Users\Zachary\phd_classes\uganda
Processed dir: C:\Users\Zachary\phd_classes\uganda\data\processed


## 1  Load data

In [4]:
# ── Grid cells ────────────────────────────────────────────────────────────────
grid = gpd.read_file(processed_dir / "palabek_1km_grids.geojson")
grid = grid.to_crs(PROJECTED_CRS)
print(f"Grid cells loaded: {len(grid)}  CRS: {grid.crs.to_epsg()}")

# ── Assignment CSVs ───────────────────────────────────────────────────────────
assign_dir = processed_dir / "assignments"
hank_assign  = pd.read_csv(assign_dir / "palabek_1km_road_intersect_hank_assignments.csv")
tanna_assign = pd.read_csv(assign_dir / "palabek_1km_road_intersect_tanna_assignments.csv")

print(f"Hank assignment rows : {len(hank_assign)}")
print(f"Tanna assignment rows: {len(tanna_assign)}")

Grid cells loaded: 1281  CRS: 32636
Hank assignment rows : 384
Tanna assignment rows: 384


In [5]:
# ── Identify overlapping cells ────────────────────────────────────────────────
# Overlap cells are those flagged is_overlap=True in BOTH assignment files.
hank_overlap_oids  = set(hank_assign.loc[hank_assign["is_overlap"], "OID"])
tanna_overlap_oids = set(tanna_assign.loc[tanna_assign["is_overlap"], "OID"])
overlap_oids = sorted(hank_overlap_oids & tanna_overlap_oids)

print(f"Hank cells with is_overlap=True  : {len(hank_overlap_oids)}")
print(f"Tanna cells with is_overlap=True : {len(tanna_overlap_oids)}")
print(f"Cells in BOTH overlap sets       : {len(overlap_oids)}")
print(f"Overlap OIDs: {overlap_oids}")

# Restrict grid to overlap cells
overlap_grid = grid[grid["OID"].isin(overlap_oids)].copy().reset_index(drop=True)
print(f"\nGrid rows matched: {len(overlap_grid)}")

Hank cells with is_overlap=True  : 44
Tanna cells with is_overlap=True : 44
Cells in BOTH overlap sets       : 22
Overlap OIDs: [621, 622, 657, 658, 659, 693, 694, 695, 729, 730, 731, 766, 767, 768, 802, 803, 804, 838, 839, 840, 874, 875]

Grid rows matched: 22


In [6]:
# ── Digitization files ────────────────────────────────────────────────────────
dig_dir = processed_dir / "sample_digitization"

hank_raw  = gpd.read_file(dig_dir / "Lawit-S2-digitize.geojson")
tanna_raw = gpd.read_file(dig_dir / "Sample_Digitization_Palabek_Tanna.geojson")

# Reproject both to UTM 36N for metric calculations
hank_lines  = hank_raw.to_crs(PROJECTED_CRS)
tanna_lines = tanna_raw.to_crs(PROJECTED_CRS)

# Explode MultiLineStrings → individual LineStrings for cleaner per-segment analysis
hank_lines  = hank_lines.explode(index_parts=False).reset_index(drop=True)
tanna_lines = tanna_lines.explode(index_parts=False).reset_index(drop=True)

print(f"Hank  lines (after explode): {len(hank_lines)}  | CRS: {hank_lines.crs.to_epsg()}")
print(f"Tanna lines (after explode): {len(tanna_lines)} | CRS: {tanna_lines.crs.to_epsg()}")

Hank  lines (after explode): 197  | CRS: 32636
Tanna lines (after explode): 1032 | CRS: 32636


## 2  Clip digitizations to overlap grid cells & compute per-cell metrics

In [8]:
CELL_AREA_KM2 = (CELL_SIZE_M / 1000) ** 2

_LINE_TYPES = ("LineString", "MultiLineString", "LinearRing")

def _count_nodes(geom) -> int:
    """Recursively count coordinate vertices in any geometry type."""
    if geom is None or geom.is_empty:
        return 0
    t = geom.geom_type
    if t in ("LineString", "LinearRing"):
        return len(geom.coords)
    if t == "Point":
        return 1
    if t in ("MultiLineString", "MultiPoint", "MultiPolygon", "GeometryCollection"):
        return sum(_count_nodes(part) for part in geom.geoms)
    if t == "Polygon":
        return len(geom.exterior.coords) + sum(len(r.coords) for r in geom.interiors)
    return 0

def _extract_lines(geom):
    """Flatten any geometry to a list of LineString/LinearRing primitives."""
    if geom is None or geom.is_empty:
        return []
    t = geom.geom_type
    if t in ("LineString", "LinearRing"):
        return [geom]
    if t == "MultiLineString":
        return list(geom.geoms)
    if t in ("GeometryCollection", "MultiPolygon", "MultiPoint"):
        result = []
        for part in geom.geoms:
            result.extend(_extract_lines(part))
        return result
    return []

def metrics_for_lines_in_cell(lines_gdf, cell_geom) -> dict:
    """Clip lines to cell and return summary metrics."""
    _empty = dict(
        n_segments=0, n_nodes=0,
        total_length_m=0.0, density_m_per_km2=0.0,
        mean_seg_length_m=float("nan"),
    )
    clipped = lines_gdf[lines_gdf.intersects(cell_geom)].copy()
    if clipped.empty:
        return _empty
    clipped["geometry"] = clipped.geometry.intersection(cell_geom)
    clipped = clipped[~clipped.geometry.is_empty].copy()
    if clipped.empty:
        return _empty

    # Flatten to individual LineStrings (intersection can return GeometryCollections)
    all_lines = []
    for geom in clipped.geometry:
        all_lines.extend(_extract_lines(geom))
    if not all_lines:
        return _empty

    lengths = np.array([g.length for g in all_lines])
    total   = lengths.sum()
    n_nodes = sum(_count_nodes(g) for g in all_lines)
    return dict(
        n_segments        = len(all_lines),
        n_nodes           = int(n_nodes),
        total_length_m    = total,
        density_m_per_km2 = total / CELL_AREA_KM2,
        mean_seg_length_m = float(lengths.mean()),
    )


rows = []
for _, cell_row in overlap_grid.iterrows():
    oid  = cell_row["OID"]
    cell = cell_row.geometry
    hm = metrics_for_lines_in_cell(hank_lines,  cell)
    tm = metrics_for_lines_in_cell(tanna_lines, cell)
    rows.append({"OID": oid, "participant": "Hank",  **hm})
    rows.append({"OID": oid, "participant": "Tanna", **tm})

cell_metrics = pd.DataFrame(rows)
print(f"Cell-level metrics computed for {len(overlap_oids)} overlap cells x 2 participants")
cell_metrics


Cell-level metrics computed for 22 overlap cells x 2 participants


,OID,participant,n_segments,n_nodes,total_length_m,density_m_per_km2,mean_seg_length_m
0,621,Hank,2,28,1176.8,1176.8,588.4
1,621,Tanna,4,41,1183.8,1183.8,295.9
2,622,Hank,3,24,1595.9,1595.9,532.0
3,622,Tanna,8,55,1274.7,1274.7,159.3
4,657,Hank,2,34,1109.5,1109.5,554.7
5,657,Tanna,7,112,3111.1,3111.1,444.4
6,658,Hank,3,13,876.7,876.7,292.2
7,658,Tanna,3,24,734.1,734.1,244.7
8,659,Hank,1,4,422.6,422.6,422.6
9,659,Tanna,6,81,1951.8,1951.8,325.3


## 3  Aggregate statistics across all overlap cells

In [9]:
agg_cols = ["total_length_m", "density_m_per_km2", "n_segments", "n_nodes", "mean_seg_length_m"]

agg = (
    cell_metrics
    .groupby("participant")[agg_cols]
    .agg(["mean", "median", "std", "min", "max"])
)
print("=== Aggregate statistics across all overlap cells ===")
display(agg)

=== Aggregate statistics across all overlap cells ===


total_length_m                           density_m_per_km2         \
                      mean median    std  min    max              mean median   
participant                                                                     
Hank                1151.7 1133.2  932.3  0.0 3356.3            1151.7 1133.2   
Tanna               1846.5 1772.9 1170.6 21.1 4179.1            1846.5 1772.9   

                               n_segments                    n_nodes         \
               std  min    max       mean median std min max    mean median   
participant                                                                   
Hank         932.3  0.0 3356.3        2.1    2.0 1.7   0   6    17.6   20.5   
Tanna       1170.6 21.1 4179.1        5.4    4.5 3.2   1  14    66.0   61.5   

                          mean_seg_length_m                            
             std min  max              mean median   std   min    max  
participant                                                            
Hank        13.5   0   39             598.4  554.7 264.7 184.4 1156.9  
Tanna       42.3   2  170             342.4  317.0 181.4  21.1  865.1

In [10]:
# ── Paired t-tests / Wilcoxon signed-rank for each metric ────────────────────
hank_vals  = cell_metrics[cell_metrics["participant"] == "Hank"].set_index("OID")
tanna_vals = cell_metrics[cell_metrics["participant"] == "Tanna"].set_index("OID")

stat_rows = []
for col in agg_cols:
    h = hank_vals.loc[overlap_oids, col].dropna().values
    t = tanna_vals.loc[overlap_oids, col].dropna().values
    # Only use paired values where both are valid
    mask = ~(np.isnan(h) | np.isnan(t))
    h, t = h[mask], t[mask]
    if len(h) < 3:
        continue
    t_stat, t_p    = stats.ttest_rel(h, t)
    w_stat, w_p    = stats.wilcoxon(h, t) if not np.all(h == t) else (np.nan, np.nan)
    stat_rows.append(dict(
        metric         = col,
        hank_mean      = h.mean(),
        tanna_mean     = t.mean(),
        diff_mean      = (h - t).mean(),
        ttest_p        = t_p,
        wilcoxon_p     = w_p,
        n_pairs        = len(h),
    ))

stat_df = pd.DataFrame(stat_rows)
print("=== Paired statistical tests (Hank vs Tanna, over overlap cells) ===")
display(stat_df.round(4))

ValueError: operands could not be broadcast together with shapes (17,) (22,) 

## 4  Per-metric comparison plots (all overlap cells)

In [ ]:
COLORS = {"Hank": "#2166ac", "Tanna": "#d6604d"}

fig, axes = plt.subplots(1, len(agg_cols), figsize=(18, 4))
fig.suptitle("Distribution of per-cell metrics across overlap cells", fontsize=12)

for ax, col in zip(axes, agg_cols):
    for participant, color in COLORS.items():
        vals = cell_metrics[cell_metrics["participant"] == participant][col].dropna()
        ax.hist(vals, bins=10, alpha=0.6, color=color, label=participant, edgecolor="white")
    ax.set_title(col.replace("_", "\n"), fontsize=9)
    ax.set_xlabel("")
    ax.tick_params(labelsize=8)

axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter: Hank vs Tanna per cell for key metrics ──────────────────────────
key_metrics = [
    ("total_length_m",    "Total road length (m)"),
    ("density_m_per_km2", "Road density (m/km²)"),
    ("n_segments",        "Number of segments"),
    ("n_nodes",           "Number of nodes"),
]

fig, axes = plt.subplots(1, len(key_metrics), figsize=(16, 4))
fig.suptitle("Per-cell comparison: Hank (x) vs Tanna (y) — each point is one overlap cell", fontsize=11)

for ax, (col, label) in zip(axes, key_metrics):
    x = hank_vals.loc[overlap_oids, col].values
    y = tanna_vals.loc[overlap_oids, col].values
    ax.scatter(x, y, color="#555555", alpha=0.7, s=50, zorder=3)
    # 1:1 reference line
    lim = max(np.nanmax(x), np.nanmax(y)) * 1.05
    ax.plot([0, lim], [0, lim], "--", color="#aaaaaa", linewidth=1, zorder=1)
    # Annotate OIDs for outliers (top 3 by absolute diff)
    diffs = np.abs(x - y)
    top_idx = np.argsort(diffs)[-3:]
    for i in top_idx:
        ax.annotate(str(overlap_oids[i]), (x[i], y[i]), fontsize=7,
                    xytext=(4, 4), textcoords="offset points")
    ax.set_xlabel(f"Hank", fontsize=9)
    ax.set_ylabel(f"Tanna", fontsize=9)
    ax.set_title(label, fontsize=9)
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Side-by-side bar chart of mean metrics per participant ────────────────────
mean_vals = cell_metrics.groupby("participant")[agg_cols].mean()

fig, axes = plt.subplots(1, len(agg_cols), figsize=(18, 4))
fig.suptitle("Mean per-cell metric across all overlap cells", fontsize=12)

for ax, col in zip(axes, agg_cols):
    bars = ax.bar(
        mean_vals.index,
        mean_vals[col],
        color=[COLORS[p] for p in mean_vals.index],
        edgecolor="white",
        width=0.5,
    )
    ax.bar_label(bars, fmt="%.0f", fontsize=8, padding=2)
    ax.set_title(col.replace("_", "\n"), fontsize=9)
    ax.tick_params(labelsize=8)
    ax.set_ylim(0, mean_vals[col].max() * 1.25)

plt.tight_layout()
plt.show()

## 5  Per-cell detail table

In [ ]:
# ── Wide-format table: one row per overlap cell, columns for each participant ─
wide = cell_metrics.pivot(index="OID", columns="participant", values=agg_cols)
wide.columns = [f"{metric}_{p.lower()}" for metric, p in wide.columns]
wide = wide.reset_index()

# Difference columns (Hank - Tanna)
for col in agg_cols:
    wide[f"{col}_diff"] = wide[f"{col}_hank"] - wide[f"{col}_tanna"]

# Sort by absolute total length difference descending
wide = wide.sort_values("total_length_m_diff", key=abs, ascending=False)

print("=== Per-cell metrics (sorted by |length diff|) ===")
display(wide.round(1))

## 6  Individual cell maps — selected overlap cells

Side-by-side geometry plots for the `N_EXAMPLE_CELLS` cells with the largest total-length discrepancy between participants.

In [ ]:
example_oids = wide.head(N_EXAMPLE_CELLS)["OID"].tolist()
print(f"Showing cells: {example_oids}")

fig, axes = plt.subplots(N_EXAMPLE_CELLS, 2, figsize=(10, N_EXAMPLE_CELLS * 4))
fig.suptitle("Individual overlap cells: Hank (left) vs Tanna (right)", fontsize=13, y=1.01)

for row_i, oid in enumerate(example_oids):
    cell_geom = overlap_grid.loc[overlap_grid["OID"] == oid, "geometry"].values[0]

    for col_i, (participant, lines_gdf, color) in enumerate([
        ("Hank",  hank_lines,  COLORS["Hank"]),
        ("Tanna", tanna_lines, COLORS["Tanna"]),
    ]):
        ax = axes[row_i, col_i]

        # Cell boundary
        gpd.GeoSeries([cell_geom]).boundary.plot(
            ax=ax, color="black", linewidth=1.5, zorder=1
        )

        # Clipped roads
        clipped = lines_gdf[lines_gdf.intersects(cell_geom)].copy()
        if not clipped.empty:
            clipped["geometry"] = clipped.geometry.intersection(cell_geom)
            clipped = clipped[~clipped.geometry.is_empty]
        if not clipped.empty:
            clipped.plot(ax=ax, color=color, linewidth=1.0, zorder=2)

        # Metrics annotation
        m = cell_metrics[
            (cell_metrics["OID"] == oid) & (cell_metrics["participant"] == participant)
        ].iloc[0]
        ax.set_title(
            f"OID {oid} — {participant}\n"
            f"{m.total_length_m:.0f} m  |  {m.n_segments} segs  |  {m.n_nodes} nodes",
            fontsize=9,
        )
        ax.set_axis_off()

plt.tight_layout()
plt.show()

## 7  Geometric agreement — buffered overlap ratio per cell

For each overlap cell, buffers both digitizations by `SNAP_BUFFER_M` metres and computes:

```
overlap_ratio = area(H_buf ∩ T_buf) / area(H_buf ∪ T_buf)
```

A ratio of 1 means perfect spatial agreement; 0 means no spatial overlap at all.

In [ ]:
def buffered_overlap_ratio(lines_h, lines_t, cell_geom, buf_m):
    """IoU of buffered road geometries clipped to cell."""
    def clipped_union(lines_gdf):
        sub = lines_gdf[lines_gdf.intersects(cell_geom)].copy()
        if sub.empty:
            return None
        sub["geometry"] = sub.geometry.intersection(cell_geom)
        sub = sub[~sub.geometry.is_empty]
        if sub.empty:
            return None
        return unary_union(sub.geometry.buffer(buf_m))

    h_poly = clipped_union(lines_h)
    t_poly = clipped_union(lines_t)
    if h_poly is None and t_poly is None:
        return np.nan
    if h_poly is None or t_poly is None:
        return 0.0
    intersection = h_poly.intersection(t_poly).area
    union        = h_poly.union(t_poly).area
    return intersection / union if union > 0 else np.nan


overlap_ratios = []
for _, cell_row in overlap_grid.iterrows():
    ratio = buffered_overlap_ratio(
        hank_lines, tanna_lines, cell_row.geometry, SNAP_BUFFER_M
    )
    overlap_ratios.append({"OID": cell_row["OID"], "overlap_ratio": ratio})

overlap_ratio_df = pd.DataFrame(overlap_ratios)
print(f"Mean buffered overlap ratio : {overlap_ratio_df.overlap_ratio.mean():.3f}")
print(f"Median buffered overlap ratio: {overlap_ratio_df.overlap_ratio.median():.3f}")
display(overlap_ratio_df.sort_values("overlap_ratio").round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
ax = axes[0]
ax.hist(overlap_ratio_df["overlap_ratio"].dropna(), bins=10, color="#555555", edgecolor="white")
ax.axvline(overlap_ratio_df["overlap_ratio"].mean(), color="red", linestyle="--",
           label=f"mean = {overlap_ratio_df.overlap_ratio.mean():.2f}")
ax.set_xlabel(f"Buffered overlap ratio ({SNAP_BUFFER_M} m buffer)")
ax.set_ylabel("Number of cells")
ax.set_title("Geometric agreement distribution")
ax.legend(fontsize=9)

# Scatter: overlap ratio vs mean total length
ax = axes[1]
mean_len = cell_metrics.groupby("OID")["total_length_m"].mean()
merged = overlap_ratio_df.set_index("OID").join(mean_len.rename("mean_length_m"))
ax.scatter(merged["mean_length_m"], merged["overlap_ratio"], color="#555555", alpha=0.7, s=50)
for oid, row in merged.iterrows():
    if row.overlap_ratio < overlap_ratio_df.overlap_ratio.quantile(0.25):
        ax.annotate(str(oid), (row.mean_length_m, row.overlap_ratio),
                    fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("Mean total road length (m)")
ax.set_ylabel(f"Overlap ratio")
ax.set_title("Geometric agreement vs road density")

plt.tight_layout()
plt.show()

## 8  Individual cell maps — lowest geometric agreement

In [ ]:
low_agreement_oids = (
    overlap_ratio_df.sort_values("overlap_ratio")
    .head(N_EXAMPLE_CELLS)["OID"]
    .tolist()
)
print(f"Lowest-agreement cells: {low_agreement_oids}")

fig, axes = plt.subplots(N_EXAMPLE_CELLS, 2, figsize=(10, N_EXAMPLE_CELLS * 4))
fig.suptitle(
    f"Lowest geometric agreement: Hank (left) vs Tanna (right)  [{SNAP_BUFFER_M} m buffer]",
    fontsize=12, y=1.01
)

for row_i, oid in enumerate(low_agreement_oids):
    cell_geom = overlap_grid.loc[overlap_grid["OID"] == oid, "geometry"].values[0]
    ratio = overlap_ratio_df.loc[overlap_ratio_df["OID"] == oid, "overlap_ratio"].values[0]

    for col_i, (participant, lines_gdf, color) in enumerate([
        ("Hank",  hank_lines,  COLORS["Hank"]),
        ("Tanna", tanna_lines, COLORS["Tanna"]),
    ]):
        ax = axes[row_i, col_i]
        gpd.GeoSeries([cell_geom]).boundary.plot(ax=ax, color="black", linewidth=1.5, zorder=1)

        clipped = lines_gdf[lines_gdf.intersects(cell_geom)].copy()
        if not clipped.empty:
            clipped["geometry"] = clipped.geometry.intersection(cell_geom)
            clipped = clipped[~clipped.geometry.is_empty]
        if not clipped.empty:
            clipped.plot(ax=ax, color=color, linewidth=1.0, zorder=2)

        m = cell_metrics[
            (cell_metrics["OID"] == oid) & (cell_metrics["participant"] == participant)
        ].iloc[0]
        ax.set_title(
            f"OID {oid} — {participant}  (IoU={ratio:.2f})\n"
            f"{m.total_length_m:.0f} m  |  {m.n_segments} segs  |  {m.n_nodes} nodes",
            fontsize=9,
        )
        ax.set_axis_off()

plt.tight_layout()
plt.show()

## 9  Summary

In [ ]:
print("=" * 60)
print("DIGITIZATION COMPARISON SUMMARY")
print(f"Settlement : Palabek  |  Grid: {CELL_SIZE_M} m  |  Overlap cells: {len(overlap_oids)}")
print("=" * 60)

for participant in ["Hank", "Tanna"]:
    sub = cell_metrics[cell_metrics["participant"] == participant]
    print(f"\n{participant}:")
    print(f"  Mean road length / cell : {sub.total_length_m.mean():>8.0f} m")
    print(f"  Mean density / cell     : {sub.density_m_per_km2.mean():>8.0f} m/km²")
    print(f"  Mean segments / cell    : {sub.n_segments.mean():>8.1f}")
    print(f"  Mean nodes / cell       : {sub.n_nodes.mean():>8.1f}")
    print(f"  Mean seg length         : {sub.mean_seg_length_m.mean():>8.0f} m")

print(f"\nGeometric agreement (buffered IoU, {SNAP_BUFFER_M} m):")
print(f"  Mean   : {overlap_ratio_df.overlap_ratio.mean():.3f}")
print(f"  Median : {overlap_ratio_df.overlap_ratio.median():.3f}")
print(f"  Min    : {overlap_ratio_df.overlap_ratio.min():.3f}  (OID {overlap_ratio_df.loc[overlap_ratio_df.overlap_ratio.idxmin(), 'OID']})")
print(f"  Max    : {overlap_ratio_df.overlap_ratio.max():.3f}  (OID {overlap_ratio_df.loc[overlap_ratio_df.overlap_ratio.idxmax(), 'OID']})")